# Change detection in optical satellite time series - Tutorial

<div align="center">
  
  <!-- Institution Logos -->
  <img src="https://www.umr-lastig.fr/static/lastig_1920_EN-5022e6685c1e4b6d7e84d2a08667be4e.png" height="60" alt="LASTIG">
  &nbsp;&nbsp;&nbsp;&nbsp;
  <img src="https://iadf-school.org/wp-content/uploads/2022/09/logo_iadfschool.png" height="60" alt="IADF School">
  &nbsp;&nbsp;&nbsp;&nbsp;
  <img src="https://design-system.ign.fr/img/styleguide/sg_logos/IGN_logo_RVB.png" height="60" alt="IGN">
  
  <br><br>
  
  <!-- Colab Badge -->
  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ElliotVincent/iadf_tutorial/blob/main/tutorial_1.ipynb)
  
  
</div>

---

## **Tutorial 1: Starting with SITS and post-classification change detection**

**Welcome to Tutorial 1!** In this hands-on session, you will learn how to use a semantic segmentation model for change detection in optical satellite time series in a post-classification manner

### What You'll Learn
- Working with a toy dataset of Planet multispectral imagery
- Visualize SITS and spectral indices (NDVI and NDWI)
- Train a semantic segmentation model
- Evaluating model performance in a post-classification framework
- Practical implementation in Google Colab

### Prerequisites
- Basic knowledge of Python and deep learning
- Familiarity with remote sensing concepts
- Google account for Colab access

---

## Introduction

This tutorial demonstrates how to perform change detection in optical satellite time series using Python.  
We will use a sample dataset of satellite images and apply various techniques to identify changes over time.  
The sample dataset is extracted from DynamicEarthNet [1] and focuses on 3 AoIs.  
The map below shows the location of the 3 AoIs in the world:  


![Map of the 3 AoIs](https://github.com/ElliotVincent/iadf_tutorial/blob/main/aois_location.png?raw=1)


In [ ]:
# Import necessary libraries
import copy
import matplotlib.pyplot as plt
import numpy as np
import random
import rasterio
import seaborn as sns
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
!pip install segmentation_models_pytorch -q
import segmentation_models_pytorch as smp

## 1. Playing with Satellite Image Time Series

### 1.1. Loading and Visualizing the Dataset

In [ ]:
DYNAMICEARTHNET_CLASSES = {
    0: {"label": "other",                     "color": (128, 128, 128)},  # grey
    1: {"label": "vegetation",                "color": ( 34, 139,  34)},  # green
    2: {"label": "water",                     "color": ( 31, 119, 180)},  # blue
}

DATES = [f"{yyyy}-{mm:02d}-01" for yyyy in range(2018, 2020) for mm in range(1, 13)]

# Utility function to perform percentile stretching on the image for better visualization
def percentile_stretch(img, low=2, high=98):
    lo, hi = np.percentile(img, (low, high))
    img_clipped = np.clip(img, lo, hi)
    return ((img_clipped - lo) / (hi - lo) * 255).astype(np.uint8)

# Function to plot a single image
def plot_image(image):
    rgb_img = image[:3]
    rgb_img = rgb_img.transpose(1, 2, 0)
    rgb_img = percentile_stretch(rgb_img)
    plt.imshow(rgb_img)
    plt.axis('off')

# Function to plot a single label
def plot_labels(label):
    rgb_label = np.zeros((label.shape[0], label.shape[1], 3), dtype=np.uint8)
    for i in range(3):
        rgb_label[label == i] = DYNAMICEARTHNET_CLASSES[i]["color"]
    plt.imshow(rgb_label)
    plt.axis('off')

# Function to plot a time series of images and labels
def plot_time_series(images, labels, num_images=10):
    num_images = min(num_images, len(images))
    fig, axes = plt.subplots(3, num_images, figsize=(int(4*num_images / 3), 4))
    for i in range(num_images):
        # Plot the image
        rgb_img = images[i, :3].transpose(1, 2, 0)
        rgb_img = percentile_stretch(rgb_img)
        axes[0, i].set_title(DATES[i], fontsize=10)
        axes[0, i].imshow(rgb_img)
        axes[0, i].axis('off')

        # Plot the semantic maps
        rgb_label = np.zeros((labels[i].shape[0], labels[i].shape[1], 3), dtype=np.uint8)
        for j in range(3):
            rgb_label[labels[i] == j] = DYNAMICEARTHNET_CLASSES[j]["color"]
        axes[1, i].imshow(rgb_label)
        axes[1, i].axis('off')

        # Plot binary change maps
        if i > 0:
            change_map = (labels[i] != labels[i-1]).astype(np.uint8) * 255
            axes[2, i].imshow(change_map, cmap='gray')
            axes[2, i].axis('off')
        else:
            axes[2, i].axis('off')
    plt.tight_layout()
    plt.show()

def load_time_series(path):
    return np.stack([rasterio.open(f"{path}/{date}.tif").read() for date in DATES], axis=0)[:, [2, 1, 0, 3]]

def load_labels(path):
    labels = np.load(f'{path}.npy')
    labels = np.where(labels == 2, 1, labels)
    labels = np.where(labels == 3, 0, labels)
    labels = np.where(labels == 4, 0, labels)
    labels = np.where(labels == 5, 2, labels)
    labels = np.where(labels == 6, 0, labels)
    return labels

We first load the sample dataset and visualize the images for each AoI.  
Satellite image time series (SITS) are stored as T x C x H x W tensors, where T is the number of time steps, C is the number of channels (spectral bands), H is the height, and W is the width of the image.  
In our case, we have 3 AoIs with T=24 (2 years of monthly images between January 2018 and December 2019), C=4 (RGB + NIR), H=1024, and W=1024.  
The spatial resolution of the images is roughly 3 meters per pixel.
DinamicEarthNet dataset is annotated with per-month semantic masks, with 7 land-cover classes: Impervious surface, Agriculture, Forest, Bare soil, Wetlands, Water, Snow/Ice.  
In this tutorial, we simplify the problem by grouping them in 3 classes:
- 0: Other (Impervious surface + Bare soil + Wetlands + Snow/Ice)
- 1: Vegetation (Agriculture + Forest)
- 2: Water (Water)

In [ ]:
!gdown 12R7wzxLnyp-jAQ_97uUqlHd8AjEyTQg5
!gdown 1q4Kx2LnId-eTPW8WZ1yJWwYujlQ2CZnj
!unzip "/content/data.zip"
!unzip "/content/labels.zip"

In [ ]:
aoi_name = '3998_3016_13'

print(f"Loading images and labels...")
images = load_time_series(aoi_name)
labels = load_labels(aoi_name)
plot_time_series(images, labels, num_images=6)


Let's check the shape of each SITS tensor.

In [ ]:
print(f"Images shape: {images.shape}, AOI 1 labels shape: {labels.shape}")


### 1.2. Playing with spectral indices

Now that we have loaded the dataset, we can start exploring the data and applying change detection techniques.  
First, let's visualize the per-class NDVI and NDWI averaged time series for the AOI 1.
NDVI (Normalized Difference Vegetation Index) and NDWI (Normalized Difference Water Index) are commonly used indices in remote sensing to assess vegetation health and water content, respectively.
They are calculated using the following formulas:
- NDVI = (NIR - Red) / (NIR + Red)
- NDWI = (Green - NIR) / (Green + NIR)

In [ ]:
binary_change_maps = (labels[1:] != labels[:-1]).sum(axis=0)
for class_id, class_info in DYNAMICEARTHNET_CLASSES.items():
    curr_pixel_indices = np.where((labels[0] == class_id) & (binary_change_maps == 0))
    pixel_time_series = images[..., curr_pixel_indices[0], curr_pixel_indices[1]]
    ndwi_time_series = (pixel_time_series[:, 1] - pixel_time_series[:, 3]) / (pixel_time_series[:, 1] + pixel_time_series[:, 3])  # NDWI = (Green - NIR) / (Green + NIR)
    ndwi_time_series_mean = ndwi_time_series.mean(axis=1)  # Average over all pixels
    ndwi_time_series_std = ndwi_time_series.std(axis=1)  # Standard deviation over all pixels
    plt.plot(DATES, ndwi_time_series_mean, color=np.array(class_info["color"])/255, label=class_info["label"])
    plt.fill_between(DATES, ndwi_time_series_mean - ndwi_time_series_std, ndwi_time_series_mean + ndwi_time_series_std, color=np.array(class_info["color"])/255, alpha=0.3)
    plt.title(f"NDWI Time Series for Average Vegetation Pixel")
plt.xlabel("Date")
plt.ylabel("NDWI")
plt.xticks(rotation=90)
plt.grid()
plt.legend(loc='upper right', labelspacing=.2, fontsize=8)
plt.show()


In [ ]:
binary_change_maps = (labels[1:] != labels[:-1]).sum(axis=0)
for class_id, class_info in DYNAMICEARTHNET_CLASSES.items():
    curr_pixel_indices = np.where((labels[0] == class_id) & (binary_change_maps == 0))
    pixel_time_series = images[..., curr_pixel_indices[0], curr_pixel_indices[1]]
    ndvi_time_series = (pixel_time_series[:, 3] - pixel_time_series[:, 0]) / (pixel_time_series[:, 3] + pixel_time_series[:, 0])  # NDVI = (NIR - Red) / (NIR + Red)
    ndvi_time_series_mean = ndvi_time_series.mean(axis=1)  # Average over all pixels
    ndvi_time_series_std = ndvi_time_series.std(axis=1)  # Standard deviation over all pixels
    plt.plot(DATES, ndvi_time_series_mean, color=np.array(class_info["color"])/255, label=class_info["label"])
    plt.fill_between(DATES, ndvi_time_series_mean - ndvi_time_series_std, ndvi_time_series_mean + ndvi_time_series_std, color=np.array(class_info["color"])/255, alpha=0.3)
    plt.title(f"NDVI Time Series for Average Vegetation Pixel")
plt.xlabel("Date")
plt.ylabel("NDVI")
plt.xticks(rotation=90)
plt.grid()
plt.legend(loc='lower right', labelspacing=.2, fontsize=8)
plt.show()


As expected, the NDWI index allows to clearly distinguish between water and non-water classes, while the NDVI also isolates the vegetation classes from the non-vegetation ones. Visualizing the NDVI and NDWI time series for a given pixel can help to identify changes in land cover over time. For example, a sudden drop in NDVI may indicate deforestation or urbanization, while a sudden increase in NDWI may indicate flooding or the creation of a new water body.  

For example, let's plot on top of the NDWI time series, the NDWI of a selected pixel in AoI 1. This pixel is located at the given coordinates (i=400, j=400), which corresponds to a coastal area.

In [ ]:
x, y = 415, 405

rgb_img = images[0, :3].transpose(1, 2, 0)
rgb_img = percentile_stretch(rgb_img)
plt.imshow(rgb_img)
plt.scatter(x, y, color='lime', marker='x', s=200, linewidth=3)
plt.title(DATES[0])
plt.axis('off')
plt.show()

In [ ]:
binary_change_maps = (labels[1:] != labels[:-1]).sum(axis=0)
for class_id, class_info in DYNAMICEARTHNET_CLASSES.items():
    curr_pixel_indices = np.where((labels[0] == class_id) & (binary_change_maps == 0))
    pixel_time_series = images[..., curr_pixel_indices[0], curr_pixel_indices[1]]
    ndwi_time_series = (pixel_time_series[:, 1] - pixel_time_series[:, 3]) / (pixel_time_series[:, 1] + pixel_time_series[:, 3])  # NDWI = (Green - NIR) / (Green + NIR)
    ndwi_time_series_mean = ndwi_time_series.mean(axis=1)  # Average over all pixels
    ndwi_time_series_std = ndwi_time_series.std(axis=1)  # Standard deviation over all pixels
    plt.plot(DATES, ndwi_time_series_mean, color=np.array(class_info["color"])/255, label=class_info["label"])
    plt.fill_between(DATES, ndwi_time_series_mean - ndwi_time_series_std, ndwi_time_series_mean + ndwi_time_series_std, color=np.array(class_info["color"])/255, alpha=0.3)
    plt.title(f"NDWI Time Series for Average Vegetation Pixel")
selected_pixel = images[..., y, x]
selected_pixel_ndwi = (selected_pixel[:, 1] - selected_pixel[:, 3]) / (selected_pixel[:, 1] + selected_pixel[:, 3])
plt.plot(DATES, selected_pixel_ndwi, color='black', label='Selected Pixel NDWI')
plt.xlabel("Date")
plt.ylabel("NDWI")
plt.xticks(rotation=90)
plt.grid()
plt.legend(loc='upper right', labelspacing=.2, fontsize=8)
plt.show()

This pixel seems to switch from a non-water class to a water class or vice versa at several time steps. This is a clear indication of a change in land cover, which can be further analyzed using change detection techniques.

In [ ]:
# Lets zoom around the selected pixel and show the images for the first 24 dates with a red cross on the selected pixel
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for i in range(24):
    rgb_img = images[i, :3].transpose(1, 2, 0)
    rgb_img = percentile_stretch(rgb_img)
    rgb_img = rgb_img[y-128:y+128, x-128:x+128, :3]
    rgb_img = percentile_stretch(rgb_img)
    axes[i // 6, i % 6].imshow(rgb_img)
    axes[i // 6, i % 6].scatter(128, 128, color='red', marker='x', s=100, linewidth=2)
    axes[i // 6, i % 6].set_title(DATES[i], fontsize=8)
    axes[i // 6, i % 6].axis('off')
plt.tight_layout()
plt.show()

### 1.3. Shaping a dataset for SITS change detection

Spatial and temporal domain shifts are known to be a major challenge in SITS change detection: they are out of the scope of this tutorial. We build a simpler setting in which each dataset split spans all 3 AoIs and both the years 2018 and 2019. Images top part are used for training, while the bottom left and bottom right are used for validation and testing respectively, as illustrated in the figure below.

![image-2.png](attachment:image-2.png)

In [ ]:
class SITSDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels, split, norm=True, num_classes=3, augment=False):
        # Load the dataset once in memory
        self.data = data
        self.labels = labels
        self.norm = norm
        # Dataset statistics for normalization, from DynamicEarthNet paper
        self.mean = torch.tensor([1042.59, 915.62, 671.26, 2605.21])
        self.std = torch.tensor([957.96, 715.55, 596.94, 1059.90])
        self.num_classes = num_classes
        self.ij_lists = {"train":[[i, j] for i in range(16) for j in range(16) if j >= 8 or i < 8],
                         "val": [[i, j] for i in range(16) for j in range(16) if i >= 12 and j < 8],
                         "test": [[i, j] for i in range(16) for j in range(16) if i >= 8 and i < 12 and j < 8]}[split]
        self.split = split
        self.augment = augment

    def __len__(self):
        return len(self.ij_lists)

    def __getitem__(self, idx):
        i, j = self.ij_lists[idx]
        x, y = i*64, j*64
        if self.split == 'train' and self.augment:
          # random crop
          x = min(x + random.randint(0, 63), 7 * 64) if j < 8 else min(x + random.randint(0, 63), 15 * 64)
          y = min(y + random.randint(0, 63), 15 * 64)
        imgs = torch.tensor(self.data[..., x:x+64, y:y+64])
        labs = torch.tensor(self.labels[..., x:x+64, y:y+64])
        if self.norm:
            imgs = (imgs - self.mean[:, None, None]) / self.std[:, None, None]
        if self.augment:
            # random rotation
            rot = random.randint(0, 3)
            imgs = torch.rot90(imgs, rot, [2, 3])
            labs = torch.rot90(labs, rot, [1, 2])
            # random flip
            flip = random.randint(0, 1)
            if flip == 1:
                imgs = torch.flip(imgs, [2])
                labs = torch.flip(labs, [1])
        return imgs, labs


In [ ]:
train_set = SITSDataset(data=images,
                        labels=labels,
                        split='train', norm=True, num_classes=3, augment=True)

val_set = SITSDataset(data=images,
                      labels=labels,
                      split='val', norm=True, num_classes=3)

test_set = SITSDataset(data=images,
                      labels=labels,
                      split='test', norm=True, num_classes=3)

print(f"Train dataset length: {len(train_set)}")
print(f"Validation dataset length: {len(val_set)}")
print(f"Test dataset length: {len(test_set)}")

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Base U-Net without classification head
        self.unet = smp.Unet(
            encoder_name="mobilenet_v2",    # Backbone for feature extraction
            encoder_weights='imagenet',           # Pre-trained on ImageNet
            in_channels=4,                  # 4 bands (R, G, B, NIR)
            classes=3
        )

    def forward(self, x):
        # Input x of shape (B, C, H, W)
        logits = self.unet(x)  # Shape: (B, 6, H, W)
        return logits

# Create the super-resolution model
model = UNet()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def compute_metrics(confusion_matrix):
    # Compute overall accuracy
    overall_accuracy = torch.trace(confusion_matrix) / torch.sum(confusion_matrix)

    # Compute mean IoU
    intersection = torch.diagonal(confusion_matrix)
    union = torch.sum(confusion_matrix, dim=1) + torch.sum(confusion_matrix, dim=0) - intersection
    mean_iou = torch.mean(intersection / union)

    return overall_accuracy, mean_iou


def training_step(batch, model, criterion, optimizer, device):
    imgs, labs = batch
    imgs = imgs.to(device)
    labs = labs.to(device)
    # imgs of shape (B, T, C, H, W)
    optimizer.zero_grad()
    temp_indices = torch.tensor([random.randint(0, imgs.size(1) - 1) for _ in range(imgs.size(0))], device=imgs.device)  # (B)
    batch_idx = torch.arange(imgs.size(0), device=imgs.device)
    imgs, labs = imgs[batch_idx, temp_indices], labs[batch_idx, temp_indices]
    logits = model(imgs)
    loss = criterion(logits, labs.long())
    loss.backward()
    optimizer.step()
    return loss


def validation_step(batch, model, criterion, device):
    imgs, labs = batch
    imgs = imgs.to(device)
    labs = labs.to(device)
    # imgs of shape (1, T, C, H, W)
    imgs, labs = imgs.reshape(-1, *imgs.shape[2:]), labs.reshape(-1, *labs.shape[2:])  # Flatten the time dimension
    logits = model(imgs)
    loss = criterion(logits, labs.long())
    pred = torch.argmax(logits, dim=1)
    return loss, pred, labs

In [ ]:
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False)
model = UNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
device = 'cuda'
model.to(device)

logs = {"train_loss": [], "val_loss": [], "val_acc": [], "val_miou": [], "learning_rate": []}
best_val_miou = 0.0
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    model.train()
    avg_train_loss = 0.0
    for batch in train_loader:
        loss = training_step(batch, model, criterion, optimizer, device)
        avg_train_loss += loss.item()
    avg_train_loss /= len(train_loader)
    print(f"   Average Training Loss: {avg_train_loss:.4f}")
    model.eval()
    avg_val_loss = 0.0
    val_conf_matrix = torch.zeros(3, 3, dtype=torch.long, device=device)
    with torch.no_grad():
        for batch in val_loader:
            loss, pred, labs = validation_step(batch, model, criterion, device)
            avg_val_loss += loss.item()
            val_conf_matrix += torch.bincount(pred.flatten().long() * 3 + labs.flatten().long(), minlength=9).reshape(3, 3)
    avg_val_loss /= len(val_loader)
    acc, miou = compute_metrics(val_conf_matrix)
    print(f"   Average Validation Loss: {avg_val_loss:.4f}")
    print(f"   Validation Accuracy: {acc*100:.2f}%, mIoU: {miou*100:.2f}%")
    logs["train_loss"].append(avg_train_loss)
    logs["val_loss"].append(avg_val_loss)
    logs["val_acc"].append(acc.item())
    logs["val_miou"].append(miou.item())
    if miou.item() > best_val_miou:
        best_val_miou = miou.item()
        best_model = copy.deepcopy(model)
        best_model_val_conf_matrix = val_conf_matrix.clone()
    logs["learning_rate"].append(optimizer.param_groups[0]['lr'])
    scheduler.step()

In [ ]:
# Visualize the trainng curves
plt.figure(figsize=(12, 6))
plt.plot(logs["train_loss"], label="Train Loss")
plt.plot(logs["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Losses")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(logs["learning_rate"], label="Learning Rate")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Learning Rate Schedule")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(logs["val_acc"], label="Validation Accuracy")
plt.plot(logs["val_miou"], label="Validation mIoU")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Validation Metrics")
plt.legend()
plt.show()

In [ ]:
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
val_conf_matrix_normalized = best_model_val_conf_matrix.float() / best_model_val_conf_matrix.sum(dim=0, keepdim=True) * 100
sns.heatmap(val_conf_matrix_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)], yticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Validation Confusion Matrix')
plt.show()

In [ ]:
canva_lab = torch.zeros(256, 512)
canva_pred = torch.zeros(256, 512)
ij_list = [[i, j] for i in range(4) for j in range(8)]
model.eval()
time_step = 6  # July 2018
for k in range(len(val_set)):
    imgs, labs = val_set[k]
    imgs = imgs.to(device)[time_step]
    with torch.no_grad():
      logits = model(imgs.unsqueeze(0))[0]
    imgs = imgs.detach().cpu()
    pred = logits.argmax(dim=0).detach().cpu()
    labs = labs.detach().cpu()[time_step]
    i, j = ij_list[k]
    canva_lab[i*64:(i+1)*64, j*64:(j+1)*64] = labs
    canva_pred[i*64:(i+1)*64, j*64:(j+1)*64] = pred

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title(f"Time Step {time_step}")
plot_image(images[time_step, :, 768:, :512])
plt.subplot(1, 3, 2)
plt.title(f"Ground Truth")
plot_labels(canva_lab.numpy())
plt.subplot(1, 3, 3)
plt.title(f"Prediction")
plot_labels(canva_pred.numpy())
plt.show()

We have trained a simple Unet model for semantic segmentation on our dataset, considering each frame independently.
We have verified that our trained model achieves correct predictions on the validation set, both quantitatively and qualitatively.  

Now, let's evaluate our model on the change detection task on the test set.

In [ ]:
canva_lab = torch.zeros(24, 256, 512)
canva_pred = torch.zeros(24, 256, 512)
ij_list = [[i, j] for i in range(4) for j in range(8)]
model.eval()
times = []
for k in range(len(test_set)):
    curr_time = 0
    for time_step in range(24):
        imgs, labs = test_set[k]
        imgs = imgs.to(device)[time_step]
        with torch.no_grad():
            start = time.time()
            logits = model(imgs.unsqueeze(0))[0]
            end = time.time()
            curr_time += end - start
        imgs = imgs.detach().cpu()
        pred = logits.argmax(dim=0).detach().cpu()
        labs = labs.detach().cpu()[time_step]
        i, j = ij_list[k]
        canva_lab[time_step, i*64:(i+1)*64, j*64:(j+1)*64] = labs
        canva_pred[time_step, i*64:(i+1)*64, j*64:(j+1)*64] = pred
    times.append(curr_time)

In [ ]:
test_confusion_matrix = torch.bincount(canva_pred.flatten().long() * 3 + canva_lab.flatten().long(), minlength=9).reshape(3, 3)
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
test_confusion_matrix_normalized = test_confusion_matrix.float() / test_confusion_matrix.sum(dim=0, keepdim=True) * 100
sns.heatmap(test_confusion_matrix_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)], yticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Test Confusion Matrix')
plt.show()

In [ ]:
test_confusion_matrix_change = torch.bincount((canva_pred[:, 1:] != canva_pred[:, :-1]).flatten().long() * 2 + (canva_lab[:, 1:] != canva_lab[:, :-1]).flatten().long(), minlength=4).reshape(2, 2)
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
test_confusion_matrix_change_normalized = test_confusion_matrix_change.float() / test_confusion_matrix_change.sum(dim=0, keepdim=True) * 100
sns.heatmap(test_confusion_matrix_change_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=['No Change', 'Change'], yticklabels=['No Change', 'Change'])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Test Confusion Matrix for Change Detection')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
for time_step in range(6):
    plt.subplot(3, 6, time_step + 1)
    plt.title(f"Time Step {time_step}")
    plot_image(images[time_step, :, 512:768, :512])
    if time_step > 0:
        plt.subplot(3, 6, time_step + 7)
        plt.title(f"Ground Truth")
        plt.imshow((canva_lab[time_step-1] != canva_lab[time_step]).numpy(), cmap='gray')
        plt.axis('off')
        plt.subplot(3, 6, time_step + 13)
        plt.title(f"Prediction")
        plt.imshow((canva_pred[time_step-1] != canva_pred[time_step]).numpy(), cmap='gray')
        plt.axis('off')
plt.show()

Looking at the predicted binary change mask, compared to the ground truth, we can observe two things:
1. Predicted change maps are very noisy, with a significant amount of false positives.  
2. Some real changes are missed by the model, leading to false negatives.  
Overall, change performance are low, as shown by the change IoU and the false positive rate.

In [ ]:
change_iou = test_confusion_matrix_change[1, 1].float() / (test_confusion_matrix_change[1, 1].float() + test_confusion_matrix_change[1, 0].float() + test_confusion_matrix_change[0, 1].float())
false_positive_rate = test_confusion_matrix_change[1, 0].float() / (test_confusion_matrix_change[1, 0].float() + test_confusion_matrix_change[0, 0].float())
acc = test_confusion_matrix.diag().sum().item() / test_confusion_matrix.sum().item()
per_class_iou = test_confusion_matrix.diag() / (test_confusion_matrix.sum(dim=1) + test_confusion_matrix.sum(dim=0) - test_confusion_matrix.diag())
miou = per_class_iou.mean().item()
print(f"Test inference time: {np.mean(times)*1e3:.1f}ms ({np.std(times)*1e3:.1f}ms) per time series")
print(f"Sem Seg Acc:         {acc*100:.1f}")
print(f"Sem Seg mIoU:        {miou*100:.1f}")
print(f"Change IoU:          {change_iou.item() * 100:.1f}")
print(f"False Positive Rate: {false_positive_rate.item() * 100:.2f}")

### Bibliography

[1] Toker et al. _DynamicEarthNet: Daily Multi-Spectral Satellite Dataset for Semantic Change Segmentation_. CVPR 2022.